In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import os
import trimesh
import numpy as np

Det 3D scanninger, hvor der både er en objekt fil, række billeder af overflade og en csv med automatiserede mål.
Outcome of interest er aterosklerose på ultralyd, som vi måler 4 steder (begge sider af hals og lyske) og som både kan bruges dikotomt (ja/nej) eller numerisk (en score af byrde af aterosklerose).


Tænker de mest komplicerede data at håndtere er 3D scanningerne. Jeg har ikke inkluderet data på outcome, da det bare er et par prøvescanninger af mig selv.
Når I er klar, er i velkomne herinde, hvor vi kan gå videre med give jer dataadgang.


In [ ]:
base_path1 = r"C:\Users\JonatanEmilSvendsen\OneDrive - Sparrow Quantum Aps\Desktop\samlet 2 scanninger\3d scanning"
base_path2 = r"C:\Users\JonatanEmilSvendsen\OneDrive - Sparrow Quantum Aps\Desktop\samlet 2 scanninger\3dscanning2"

csv_path = os.path.join(base_path1, "measures.csv")

# 1) Read the entire file as lines. Adjust encoding if needed (utf-16 is common if you see 0xFFFE).
with open(csv_path, 'r', encoding='utf-16') as f:
    lines = f.read().splitlines()

# 2) The first 4 lines are just metadata. You can store or ignore them:
metadata_lines = lines[:4]
print("Metadata lines:")
for meta in metadata_lines:
    print("  ", meta)

# 3) The important part is the last 3 lines:
data_lines = lines[4:]  # lines[4], lines[5], lines[6]

# Parse each line by splitting on commas:
split_lines = [line.split(',') for line in data_lines]

# According to your snippet, we have:
#   line 5 = measurement names
#   line 6 = measurement IDs
#   line 7 = actual measurements
measurement_names = split_lines[0]
measurement_ids   = split_lines[1]
measurement_vals  = split_lines[2]

# 4) Create a multi-level column index in Pandas:
cols = pd.MultiIndex.from_arrays([measurement_names, measurement_ids])
df = pd.DataFrame([measurement_vals], columns=cols)

print("\nFinal DataFrame with Multi-Level Columns:")
print(df)

# print all measurements in a table
print(df.to_string())

# -------------------
# 1) Get the DataFrame (already parsed).
#    In your code, you already have `df`.
#    We'll assume it's called `df` with shape (N, M).
# -------------------
# Example: Suppose you followed your steps to get `df` (with the multi-level columns).
# We'll flatten the columns for convenience.

df_flat = df.copy()
df_flat.columns = ["_".join(col) for col in df_flat.columns]
# Now columns look like "Measurement Names_Body height (cm)" etc.

# -------------------
# 1) Get the DataFrame (already parsed).
#    In your code, you already have `df`.
# -------------------
df_flat = df.copy()
df_flat.columns = ["_".join(col) for col in df_flat.columns]  # Flatten multi-index column names

# Ensure numeric conversion (this filters out any non-numeric columns automatically)
df_numeric = df_flat.apply(pd.to_numeric, errors='coerce')  # Convert everything to float, NaN for non-convertible

# Fill NaN values (if any), as tensors can't handle NaN
df_numeric = df_numeric.fillna(0.0)  # Or use another imputation method if needed

# -------------------
# 2) Define a custom Dataset
# -------------------
class MeasurementsDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.tensor_data = torch.tensor(self.dataframe.values, dtype=torch.float32)  # Now fully numeric

    def __len__(self):
        return len(self.tensor_data)

    def __getitem__(self, idx):
        return self.tensor_data[idx]

# -------------------
# 3) Create the Dataset & DataLoader
# -------------------
dataset = MeasurementsDataset(df_numeric)  # Use cleaned numeric dataframe
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

# -------------------
# 4) Example usage
# -------------------
for batch in dataloader:
    print("Batch shape:", batch.shape)  # Should be [1, M] if batch_size=1
    print("Values:", batch)


In [ ]:
csv_path = os.path.join(base_path2, "measures.csv")

# 1) Read the entire file as lines. Adjust encoding if needed (utf-16 is common if you see 0xFFFE).
with open(csv_path, 'r', encoding='utf-16') as f:
    lines = f.read().splitlines()

# 2) The first 4 lines are just metadata. You can store or ignore them:
metadata_lines = lines[:4]
print("Metadata lines:")
for meta in metadata_lines:
    print("  ", meta)

# 3) The important part is the last 3 lines:
data_lines = lines[4:]  # lines[4], lines[5], lines[6]

# Parse each line by splitting on commas:
split_lines = [line.split(',') for line in data_lines]

# According to your snippet, we have:
#   line 5 = measurement names
#   line 6 = measurement IDs
#   line 7 = actual measurements
measurement_names = split_lines[0]
measurement_ids   = split_lines[1]
measurement_vals  = split_lines[2]

# 4) Create a multi-level column index in Pandas:
cols = pd.MultiIndex.from_arrays([measurement_names, measurement_ids])
df = pd.DataFrame([measurement_vals], columns=cols)

print("\nFinal DataFrame with Multi-Level Columns:")
print(df)

# print all measurements in a table
print(df.to_string())

# -------------------
# 1) Get the DataFrame (already parsed).
#    In your code, you already have `df`.
#    We'll assume it's called `df` with shape (N, M).
# -------------------
# Example: Suppose you followed your steps to get `df` (with the multi-level columns).
# We'll flatten the columns for convenience.

df_flat = df.copy()
df_flat.columns = ["_".join(col) for col in df_flat.columns]
# Now columns look like "Measurement Names_Body height (cm)" etc.

# -------------------
# 1) Get the DataFrame (already parsed).
#    In your code, you already have `df`.
# -------------------
df_flat = df.copy()
df_flat.columns = ["_".join(col) for col in df_flat.columns]  # Flatten multi-index column names

# Ensure numeric conversion (this filters out any non-numeric columns automatically)
df_numeric = df_flat.apply(pd.to_numeric, errors='coerce')  # Convert everything to float, NaN for non-convertible

# Fill NaN values (if any), as tensors can't handle NaN
df_numeric = df_numeric.fillna(0.0)  # Or use another imputation method if needed

# -------------------
# 2) Define a custom Dataset
# -------------------
class MeasurementsDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.tensor_data = torch.tensor(self.dataframe.values, dtype=torch.float32)  # Now fully numeric

    def __len__(self):
        return len(self.tensor_data)

    def __getitem__(self, idx):
        return self.tensor_data[idx]

# -------------------
# 3) Create the Dataset & DataLoader
# -------------------
dataset = MeasurementsDataset(df_numeric)  # Use cleaned numeric dataframe
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

# -------------------
# 4) Example usage
# -------------------
for batch in dataloader:
    print("Batch shape:", batch.shape)  # Should be [1, M] if batch_size=1
    print("Values:", batch)


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
obj_path = os.path.join(base_path1, "Scan.obj")
mesh = trimesh.load(obj_path)

# Check basic info
print(f"Vertices: {mesh.vertices.shape}")
print(f"Faces: {mesh.faces.shape}")

# Compute surface area and volume
surface_area = mesh.area
volume = mesh.volume
print(f"Surface Area: {surface_area}")
print(f"Volume: {volume}")

# Check if mesh is watertight
is_watertight = mesh.is_watertight
print(f"Is the mesh watertight? {is_watertight}")

# Get face normals
face_normals = mesh.face_normals
print(f"Face normals shape: {face_normals.shape}")

# Visualize mesh (requires a suitable environment)
# mesh.show()

# Example of applying transformations (e.g., rotate around Z-axis by 45 degrees)
transform_matrix = trimesh.transformations.rotation_matrix(
    np.radians(45), [0, 0, 1]
)
mesh.apply_transform(transform_matrix)
print("Rotated the mesh by 45 degrees around Z-axis.")

# Visualize the mesh
mesh.show()


In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.pyplot as plt

obj_path = os.path.join(base_path1, "Scan.obj")
mesh = trimesh.load(obj_path)

# Check basic info
print(f"Vertices: {mesh.vertices.shape}")
print(f"Faces: {mesh.faces.shape}")

# Compute surface area and volume
surface_area = mesh.area
volume = mesh.volume
print(f"Surface Area: {surface_area}")
print(f"Volume: {volume}")

# Check if mesh is watertight
is_watertight = mesh.is_watertight
print(f"Is the mesh watertight? {is_watertight}")

# Apply rotation around Z-axis by 45 degrees
transform_matrix = trimesh.transformations.rotation_matrix(np.radians(45), [0, 0, 1])
mesh.apply_transform(transform_matrix)

# Plot and save mesh as an image
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')
ax.add_collection3d(Poly3DCollection(mesh.triangles, alpha=0.7, facecolor='cyan'))

# Auto-scale axes to mesh size
scale = mesh.bounds.flatten()
ax.auto_scale_xyz(scale, scale, scale)

# Setting view angle for better visualization
ax.view_init(elev=30, azim=45)

plt.axis('off')
plt.tight_layout()

# Save figure
output_path = os.path.join(base_path1, "mesh_visualization.png")
plt.savefig(output_path, dpi=300)
plt.close()

print(f"Mesh visualization saved at {output_path}")

In [ ]:
obj_path = os.path.join(base_path2, "Scan.obj")
mesh = trimesh.load(obj_path)

# Check basic info
print(f"Vertices: {mesh.vertices.shape}")
print(f"Faces: {mesh.faces.shape}")

# Compute surface area and volume
surface_area = mesh.area
volume = mesh.volume
print(f"Surface Area: {surface_area}")
print(f"Volume: {volume}")

# Check if mesh is watertight
is_watertight = mesh.is_watertight
print(f"Is the mesh watertight? {is_watertight}")

# Get face normals
face_normals = mesh.face_normals
print(f"Face normals shape: {face_normals.shape}")

# Visualize mesh (requires a suitable environment)
# mesh.show()

# Example of applying transformations (e.g., rotate around Z-axis by 45 degrees)
transform_matrix = trimesh.transformations.rotation_matrix(
    np.radians(45), [0, 0, 1]
)
mesh.apply_transform(transform_matrix)
print("Rotated the mesh by 45 degrees around Z-axis.")

In [ ]:
# Visualize the mesh
mesh.show()